# CPDS-AI: Audio Classification (Baby Cry vs Car Noise)

This notebook is designed for Kaggle or Google Colab. The workflow includes:
1. Loading an audio dataset (Donate-a-Cry and ESC-50)
2. Extracting Mel-spectrogram features
3. Training a lightweight CNN (ResNet18)
4. Exporting an ONNX model.

In [ ]:
!pip install -q librosa onnx soundfile
import librosa
import torch
import torch.nn as nn
import torchvision.models as models
import json
from pathlib import Path
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

## 1. Audio Preprocessing (Audio to Mel-Spectrogram)
The one-dimensional waveform is converted into a two-dimensional Mel-spectrogram.

In [ ]:
def extract_mel_spectrogram(audio_path, sr=16000, duration=2.0):
    """
    Load an audio file and create a fixed-size Mel-spectrogram.
    """
    y, sr = librosa.load(audio_path, sr=sr, duration=duration)
    
    # Ensure a fixed waveform length.
    target_length = int(sr * duration)
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]
        
    # Compute the Mel-spectrogram.
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize to [0, 1] and reshape for a one-channel CNN.
    mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-6)
    return np.expand_dims(mel_spec_db, axis=0)  # Shape: (1, 128, T)

## 2. Model Definition (Compact ResNet18)

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(AudioCNN, self).__init__()
        # Adapt ResNet18 to accept one channel instead of three RGB channels.
        self.model = models.resnet18(weights=None)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Replace the classifier with two classes: baby cry and noise.
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

print("Model definition is ready.")

## 3. Training

Attach a Kaggle Dataset with `train/cry`, `train/noise` and optional `val/cry`, `val/noise` directories, then set `DATASET_DIR`. The class order is saved next to the model so inference never guesses class indices.

In [ ]:
DATASET_DIR = Path("/kaggle/input/cpds-audio")  # Update this to your Kaggle Dataset mount.
CLASS_NAMES = ["noise", "cry"]
SAMPLE_RATE, DURATION, BATCH_SIZE, EPOCHS = 16000, 2.0, 32, 20

class AudioDataset(Dataset):
    extensions = {".wav", ".mp3", ".flac", ".ogg"}
    def __init__(self, root, class_names):
        self.samples = [(path, index) for index, label in enumerate(class_names)
                        for path in (root / label).rglob("*") if path.suffix.lower() in self.extensions]
        if not self.samples:
            raise ValueError(f"No audio files found under {root}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        path, label = self.samples[index]
        return torch.from_numpy(extract_mel_spectrogram(path)).float(), label

train_root = DATASET_DIR / "train"
if not train_root.is_dir():
    raise FileNotFoundError(f"Set DATASET_DIR correctly; missing {train_root}")
train_dataset = AudioDataset(train_root, CLASS_NAMES)
val_root = DATASET_DIR / "val"
if val_root.is_dir():
    val_dataset = AudioDataset(val_root, CLASS_NAMES)
else:
    train_size = int(0.8 * len(train_dataset))
    train_dataset, val_dataset = random_split(train_dataset, [train_size, len(train_dataset) - train_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioCNN(num_classes=len(CLASS_NAMES)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
best_accuracy = -1.0

for epoch in range(EPOCHS):
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(inputs.to(device)), targets.to(device))
        loss.backward()
        optimizer.step()
    model.eval(); correct = total = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            predictions = model(inputs.to(device)).argmax(dim=1).cpu()
            correct += (predictions == targets).sum().item(); total += len(targets)
    accuracy = correct / max(total, 1)
    print(f"Epoch {epoch + 1}/{EPOCHS}: validation accuracy={accuracy:.3f}")
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), "audio_model.pth")

Path("audio_labels.json").write_text(json.dumps(CLASS_NAMES), encoding="utf-8")
print(f"Saved best model (validation accuracy {best_accuracy:.3f}) and labels.")

## 4. ONNX Export

In [ ]:
# Load best weights
model.load_state_dict(torch.load('audio_model.pth'))
model.eval()

# Example input for model tracing: (batch, channel, mels, time_steps).
dummy_input = torch.randn(1, 1, 128, 63)

# Export to ONNX.
torch.onnx.export(model,
                  dummy_input,
                  "audio_model.onnx",
                  export_params=True,
                  opset_version=11,
                  do_constant_folding=True,
                  input_names=['input'],
                  output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size', 3: 'time'},
                                'output': {0: 'batch_size'}})

print("Model exported to: audio_model.onnx")